In [3]:
import os
import cv2
import numpy as np
from tqdm import tqdm

In [4]:
RAW_PATH = r"E:\Final_Year_Project\data\raw\grade_dataset"
PROCESSED_PATH = r"E:\Final_Year_Project\data\processed\grade_dataset"

os.makedirs(PROCESSED_PATH, exist_ok=True)

classes = ["grade1", "grade2", "grade3"]

for cls in classes:
    os.makedirs(os.path.join(PROCESSED_PATH, cls), exist_ok=True)

print("Folders ready.")

Folders ready.


In [5]:
def remove_black_border(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 10, 255, cv2.THRESH_BINARY)

    coords = cv2.findNonZero(thresh)

    if coords is not None:
        x, y, w, h = cv2.boundingRect(coords)
        cropped = img[y:y+h, x:x+w]
    else:
        cropped = img

    return cropped

In [6]:
IMG_SIZE = 224
total_images = 0

for cls in classes:
    input_folder = os.path.join(RAW_PATH, cls)
    output_folder = os.path.join(PROCESSED_PATH, cls)
    
    for file in tqdm(os.listdir(input_folder), desc=f"Processing {cls}"):
        
        if file.lower().endswith((".png", ".jpg", ".jpeg")):
            img_path = os.path.join(input_folder, file)
            img = cv2.imread(img_path)

            if img is None:
                continue

            img = remove_black_border(img)
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

            save_path = os.path.join(output_folder, file)
            cv2.imwrite(save_path, img)

            total_images += 1

print("Total processed images:", total_images)

Processing grade3: 100%|█████████████████████████████████████████████████████████████| 297/297 [01:10<00:00,  4.18it/s]

Total processed images: 922


In [7]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.models as models

from torch.utils.data import Dataset, DataLoader
from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [8]:
DATASET_PATH = PROCESSED_PATH

image_paths = []
labels = []

class_mapping = {
    "grade1": 0,
    "grade2": 1,
    "grade3": 2
}

for class_name in class_mapping:
    class_dir = os.path.join(DATASET_PATH, class_name)
    
    for file in os.listdir(class_dir):
        if file.lower().endswith((".png", ".jpg", ".jpeg")):
            image_paths.append(os.path.join(class_dir, file))
            labels.append(class_mapping[class_name])

print("Total images:", len(image_paths))

Total images: 922


In [9]:
train_paths, test_paths, train_labels, test_labels = train_test_split(
    image_paths,
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels
)

print("Train size:", len(train_paths))
print("Test size:", len(test_paths))

Train size: 737
Test size: 185


In [10]:
class PhenotypeDataset(Dataset):
    def __init__(self, image_paths, labels, transform):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform
        
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert("RGB")
        img = self.transform(img)
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return img, label

In [11]:
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),

    transforms.ColorJitter(
        brightness=0.1,
        contrast=0.1,
        saturation=0.1,
        hue=0.02
    ),

    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_dataset = PhenotypeDataset(train_paths, train_labels, train_transform)
test_dataset = PhenotypeDataset(test_paths, test_labels, test_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [12]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.resnet50(pretrained=True)

for param in model.parameters():
    param.requires_grad = True

model.fc = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(model.fc.in_features, 3)
)

model = model.to(device)

E:\Final_Year_Project\gnn_env\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
E:\Final_Year_Project\gnn_env\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [13]:
weights = torch.tensor([1.0, 1.0, 1.4]).to(device)

criterion = nn.CrossEntropyLoss(weight=weights)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.00005   # LOWER LR for ResNet50
)

scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=5,
    gamma=0.5
)

In [14]:
EPOCHS = 25

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for images, labels_batch in tqdm(train_loader):
        
        images = images.to(device)
        labels_batch = labels_batch.to(device)
        
        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels_batch)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels_batch).sum().item()
        total += labels_batch.size(0)
    
    scheduler.step()
    
    accuracy = 100 * correct / total
    print(f"Epoch {epoch+1} | Loss: {total_loss:.4f} | Train Accuracy: {accuracy:.2f}%")

100%|██████████████████████████████████████████████████████████████████████████████████| 24/24 [04:11<00:00, 10.48s/it]


Epoch 1 | Loss: 25.3075 | Train Accuracy: 42.88%


100%|██████████████████████████████████████████████████████████████████████████████| 24/24 [7:44:40<00:00, 1161.70s/it]


Epoch 2 | Loss: 22.0459 | Train Accuracy: 55.77%


100%|██████████████████████████████████████████████████████████████████████████████████| 24/24 [04:02<00:00, 10.11s/it]


...

Epoch 23 | Loss: 5.7637 | Train Accuracy: 93.62%


100%|██████████████████████████████████████████████████████████████████████████████████| 24/24 [03:32<00:00,  8.85s/it]


Epoch 24 | Loss: 5.7832 | Train Accuracy: 92.81%


100%|██████████████████████████████████████████████████████████████████████████████████| 24/24 [03:13<00:00,  8.06s/it]

Epoch 25 | Loss: 6.5858 | Train Accuracy: 90.50%


In [15]:
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels_batch in test_loader:
        
        images = images.to(device)
        labels_batch = labels_batch.to(device)
        
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels_batch.cpu().numpy())

acc = accuracy_score(all_labels, all_preds)
print("✅ Test Accuracy:", acc)

print("\nClassification Report:")
print(classification_report(all_labels, all_preds))

cm = confusion_matrix(all_labels, all_preds)
print("\nConfusion Matrix:\n", cm)

✅ Test Accuracy: 0.6972972972972973

Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.75      0.80        52
           1       0.62      0.74      0.68        73
           2       0.69      0.60      0.64        60

    accuracy                           0.70       185
   macro avg       0.72      0.70      0.70       185
weighted avg       0.71      0.70      0.70       185


Confusion Matrix:
 [[39 11  2]
 [ 5 54 14]
 [ 2 22 36]]
